# Customer Segmentation using K-Means Clustering

**Project:** Data Science on Marketing ML  
**Author:** Adeniran Akinlayo Moses  
**School:** Facultet School  

## Objective
Group customers into meaningful segments based on their purchasing behaviour using the RFM framework:
- **R**ecency — how recently a customer made a purchase
- **F**requency — how often they purchase
- **M**onetary — how much they spend

## Dataset
We use the [Online Retail Dataset](https://www.kaggle.com/datasets/vijayuv/onlineretail) from Kaggle — a real transactional dataset from a UK-based online retailer.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Display settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('Libraries loaded successfully!')

## 2. Load the Dataset

In [ ]:
# Load dataset (download from Kaggle and place in the data/ folder)
df = pd.read_csv('data/OnlineRetail.csv', encoding='ISO-8859-1')

print(f'Dataset shape: {df.shape}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print('Dataset Info:')
print(df.info())
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Basic statistics
df.describe()

## 4. Data Cleaning

In [ ]:
# Remove rows with missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancelled orders (InvoiceNo starting with 'C')
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative quantities and prices
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# Convert InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice column
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f'Cleaned dataset shape: {df.shape}')
df.head()

## 5. RFM Analysis

In [ ]:
# Reference date: one day after the last transaction
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

# Compute RFM metrics per customer
rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

print('RFM Table:')
rfm.head(10)

In [ ]:
# Visualise RFM distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm['Recency'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')

axes[1].hist(rfm['Frequency'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of purchases')

axes[2].hist(rfm['Monetary'], bins=30, color='seagreen', edgecolor='white')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total spend')

plt.tight_layout()
plt.show()

## 6. Feature Scaling

In [ ]:
# Scale RFM features so K-Means isn't biased by large values
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print('Scaled features (first 5 rows):')
print(rfm_scaled[:5])

## 7. Finding the Optimal Number of Clusters (Elbow Method)

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

# Plot elbow curve
plt.plot(k_range, inertia, marker='o', color='steelblue')
plt.title('Elbow Method — Optimal Number of Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.show()

## 8. Train K-Means Model

In [ ]:
# Train with k=4 (adjust based on elbow curve)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Silhouette score (closer to 1 = better)
score = silhouette_score(rfm_scaled, rfm['Cluster'])
print(f'Silhouette Score: {score:.3f}')

rfm.head(10)

## 9. Analyse & Visualise Segments

In [ ]:
# Segment summary
segment_summary = rfm.groupby('Cluster').agg(
    Count=('CustomerID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Frequency=('Frequency', 'mean'),
    Avg_Monetary=('Monetary', 'mean')
).round(1)

print('Segment Summary:')
segment_summary

In [ ]:
# Scatter plot: Recency vs Monetary, coloured by cluster
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    rfm['Recency'], rfm['Monetary'],
    c=rfm['Cluster'], cmap='tab10', alpha=0.6, s=30
)
plt.colorbar(scatter, label='Cluster')
plt.title('Customer Segments: Recency vs Monetary Value')
plt.xlabel('Recency (days)')
plt.ylabel('Monetary Value (£)')
plt.show()

## 10. Conclusions

Based on the segmentation, we identified **4 customer groups**:

| Cluster | Description | Marketing Action |
|---|---|---|
| 0 | High value, frequent buyers | Loyalty rewards, VIP offers |
| 1 | Recent but low spend | Upsell campaigns |
| 2 | Churning customers | Re-engagement emails |
| 3 | Low engagement | Win-back discounts |

*(Update this table based on your actual cluster analysis results)*

## Next Steps
- Try different values of k and compare silhouette scores
- Add more features (product categories, location)
- Build a churn prediction model on top of these segments